# Environment-Conditioned Regional Thermal Calibration
## From Multi-Temporal UAV Thermography to Predictive Simulation

This notebook implements the core scientific contribution:

**The calibration function incorporates environmental parameters directly:**

$$\hat{Y}_k = \beta_0^{(k)} + \beta_1^{(k)} T_{amb} + \beta_2^{(k)} S + \beta_3^{(k)} W + \beta_4^{(k)} H + \beta_5^{(k)} T_{amb} \cdot S$$

where $k$ indexes the thermal region, and the interaction term $T_{amb} \cdot S$ captures how ambient temperature and solar loading jointly affect surface temperature — a physically meaningful coupling.

**Why regions, not pixels:**
- Individual pixels are noisy and sensor-gain-dependent
- Regions aggregate physically similar surfaces (same material, orientation, exposure)
- Region-level statistics are robust to registration errors across flights
- The model learns *how a type of surface responds to weather*, not how one pixel changes


In [1]:
# ============================================================
# SETUP
# ============================================================
import subprocess, sys, warnings, os, json, zipfile, pickle
from pathlib import Path
import numpy as np

for pkg in ["laspy","lazrs","plotly","scikit-learn","matplotlib","scipy"]:
    try: __import__(pkg.replace("-","_"))
    except: subprocess.run([sys.executable,"-m","pip","install",pkg,"-q"])

import laspy
import plotly.graph_objects as go
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.spatial import cKDTree
from google.colab import files

warnings.filterwarnings("ignore")
DATA = Path("calib_data"); DATA.mkdir(exist_ok=True)
print("Ready.")


Ready.


In [ ]:
# ============================================================
# UPLOAD STEP 2 OUTPUT
# ============================================================
print("Upload your Step 2/3 output ZIP:\n")
up = files.upload()
fn, ct = next(iter(up.items()))
zp_path = DATA/fn
with open(zp_path,"wb") as f: f.write(ct)
with zipfile.ZipFile(zp_path,"r") as zf: zf.extractall(DATA)
zp_path.unlink()

# Find files
zone_profiles_path = zone_labels_path = kmeans_model_path = None
for p in DATA.rglob("*"):
    if p.name=="zone_profiles.json": zone_profiles_path=p
    elif p.name=="zone_labels.json": zone_labels_path=p
    elif p.name=="kmeans_model.pkl": kmeans_model_path=p

with open(zone_profiles_path) as f: zone_profiles=json.load(f)
with open(zone_labels_path) as f: zone_labels={int(k):v for k,v in json.load(f).items()}
with open(kmeans_model_path,"rb") as f: kb=pickle.load(f)
global_model=kb["model"]; cluster_cfg=kb.get("config",{})
old_to_new=kb.get("old_to_new",{}); K=kb.get("optimal_k",4)

print(f"Zones: K={K}")
for z in range(K): print(f"  {z}: {zone_labels[z]}")
print("\nLoaded.")


Upload your Step 2/3 output ZIP:



In [ ]:
# ============================================================
# UPLOAD POINT CLOUD
# ============================================================
print("Upload your .laz or .las:\n")
up2 = files.upload()
fn2, ct2 = next(iter(up2.items()))
pc_file = DATA/fn2
with open(pc_file,"wb") as f: f.write(ct2)

las = laspy.read(str(pc_file))
bm = las.classification==6
xb=las.x[bm].copy(); yb=las.y[bm].copy(); zb=las.z[bm].copy()
rb=las.red[bm].astype(np.float32); thermal_raw=rb/rb.max()

# Downsample
N=150000; n=len(xb)
if n>N: idx=np.random.RandomState(42).choice(n,N,replace=False)
else: idx=np.arange(n)
xp=xb[idx]-xb[idx].mean(); yp=yb[idx]-yb[idx].mean(); zp_pts=zb[idx]-zb[idx].min()
tp=thermal_raw[idx]

# Assign zones via KMeans
tree=cKDTree(np.column_stack([xp,yp,zp_pts]))
_,nn_idx=tree.query(np.column_stack([xp,yp,zp_pts]),k=15)
local_std=tp[nn_idx].std(axis=1)
nx=(xp-xp.min())/(xp.max()-xp.min()+1e-8)
ny=(yp-yp.min())/(yp.max()-yp.min()+1e-8)
iw=cluster_cfg.get("intensity_weight",1.0)
tw=cluster_cfg.get("texture_weight",0.7)
sw=cluster_cfg.get("spatial_weight",0.3)
feats=np.column_stack([tp*iw,local_std*tw,nx*sw,ny*sw]).astype(np.float64)
global_model.cluster_centers_=global_model.cluster_centers_.astype(np.float64)
raw_z=global_model.predict(feats)
zones=np.vectorize(old_to_new.get)(raw_z).astype(np.uint8)

# Remove outliers
x_med,y_med=np.median(xp),np.median(yp)
x_std,y_std=np.std(xp),np.std(yp)
inlier=(np.abs(xp-x_med)<1.8*x_std)&(np.abs(yp-y_med)<1.8*y_std)&(zp_pts>0.5)
xp=xp[inlier];yp=yp[inlier];zp_pts=zp_pts[inlier];tp=tp[inlier];zones=zones[inlier]

print(f"Building: {n:,} -> {inlier.sum():,} clean points")
for z in range(K): print(f"  Zone {z} ({zone_labels[z][:25]}): {(zones==z).sum():,}")
print("\nReady.")


---
## Step 1: Build the Calibration Dataset

For each **region** (zone) $k$ and each **flight** $f$, we have:
- **Response**: mean normalized thermal intensity $\bar{Y}_k^{(f)}$
- **Predictors**: environmental vector $\mathbf{e}^{(f)} = [T, S, W, H]^\top$

This gives $K \times F = 4 \times 4 = 16$ calibration samples — small, but physically constrained.


In [ ]:
# ============================================================
# BUILD CALIBRATION DATASET
# ============================================================

calib_data = []
flight_ids = []
flight_envs = []

for z_str, profile in zone_profiles.items():
    z = int(z_str)
    for fe in profile["flights"]:
        mi = fe.get("mean_intensity")
        t  = fe.get("env_temp")
        s  = fe.get("env_solar")
        w  = fe.get("env_wind")
        h  = fe.get("env_rh")
        fid = fe.get("flight_id","?")

        if mi is not None and t is not None:
            calib_data.append({
                "zone":z, "intensity":mi,
                "temp":t, "solar":s or 500, "wind":w or 5, "rh":h or 50,
                "flight_id": fid
            })
            if z == 0:
                flight_ids.append(fid)
                flight_envs.append([t, s or 500, w or 5, h or 50])

flight_envs = np.array(flight_envs)
n_flights = len(flight_ids)

zones_cal = np.array([r["zone"] for r in calib_data])
y_cal     = np.array([r["intensity"] for r in calib_data])
env_cal   = np.array([[r["temp"],r["solar"],r["wind"],r["rh"]] for r in calib_data])

print(f"Calibration dataset: {len(calib_data)} samples ({K} zones × {n_flights} flights)")
print(f"\nFlights:")
for i,fid in enumerate(flight_ids):
    v = flight_envs[i]
    print(f"  {fid}: T={v[0]:.0f}°F  S={v[1]:.0f}W/m²  W={v[2]:.0f}mph  RH={v[3]:.0f}%")

print(f"\nZone × Flight intensity matrix:")
print(f"{'Zone':>6}", end="")
for fid in flight_ids: print(f" {fid:>10}", end="")
print()
for z in range(K):
    print(f"  Z{z}  ", end="")
    for i in range(n_flights):
        idx = z * n_flights + i
        print(f" {y_cal[idx]:>10.4f}", end="")
    print(f"  ({zone_labels[z][:20]})")


---
## Step 2: Environment-Conditioned Calibration Model

For each region $k$, the calibration function is:

$$\hat{Y}_k(\mathbf{e}) = \beta_0^{(k)} + \beta_1^{(k)} \hat{T} + \beta_2^{(k)} \hat{S} + \beta_3^{(k)} \hat{W} + \beta_4^{(k)} \hat{H} + \beta_5^{(k)} (\hat{T} \cdot \hat{S})$$

where $\hat{T} = (T - \bar{T})/\sigma_T$ are standardized inputs, and the **interaction term** $\hat{T} \cdot \hat{S}$ captures the physically meaningful coupling between ambient temperature and solar irradiance (sun-heated surfaces respond differently at different ambient temperatures).

We use Ridge regression ($\alpha=1.0$) to regularize with only 4 samples per zone.


In [ ]:
# ============================================================
# FIT CALIBRATION MODELS
# ============================================================

models = {}  # per-zone models
env_names = ["T_amb", "Solar", "Wind", "RH", "T×S"]

print("ENVIRONMENT-CONDITIONED CALIBRATION")
print("="*70)

for z in range(K):
    mask = zones_cal == z
    X_z = env_cal[mask].copy()
    y_z = y_cal[mask].copy()

    # Add interaction term: T × Solar (physically meaningful)
    T_x_S = (X_z[:, 0] * X_z[:, 1]).reshape(-1, 1)
    X_aug = np.hstack([X_z, T_x_S])

    # Standardize
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X_aug)

    # Ridge regression
    model = Ridge(alpha=1.0)
    model.fit(X_std, y_z)
    y_pred = model.predict(X_std)

    r2 = r2_score(y_z, y_pred)
    rmse = np.sqrt(mean_squared_error(y_z, y_pred))

    models[z] = {
        "model": model,
        "scaler": scaler,
        "r2": r2,
        "rmse": rmse,
        "coefs": model.coef_,
        "intercept": model.intercept_,
    }

    label = zone_labels[z]
    print(f"\nRegion {z}: {label}")
    print(f"  R² = {r2:.4f}    RMSE = {rmse:.6f}")
    print(f"  Ŷ_{z} = {model.intercept_:.4f}", end="")
    for j, name in enumerate(env_names):
        print(f" {model.coef_[j]:+.4f}·{name}", end="")
    print()
    print(f"  (standardized coefficients — magnitude = importance)")

print("\n" + "="*70)

# ── Define the calibration function ──
def calibrate(zone_id, temp, solar, wind, rh):
    """
    Environment-conditioned calibration function.
    Y_calibrated = f(zone, T_ambient, Solar, Wind, RH)
    """
    m = models[zone_id]
    X = np.array([[temp, solar, wind, rh, temp*solar]])
    X_std = m["scaler"].transform(X)
    return float(np.clip(m["model"].predict(X_std)[0], 0, 1))

def calibrate_all(temp, solar, wind, rh):
    return np.array([calibrate(z, temp, solar, wind, rh) for z in range(K)])

print("\nCalibration function: calibrate(zone, T, Solar, Wind, RH)")
print("Test: calibrate_all(70, 600, 5, 40) =", calibrate_all(70, 600, 5, 40).round(4))


---
## Publication Figures
### Figure A: Model Validation — Predicted vs Observed + Sensitivity Analysis


In [ ]:
# ============================================================
# FIGURE: MODEL VALIDATION (2×2 publication layout)
# ============================================================

ZC = ["#1b1464","#2980b9","#27ae60","#f39c12","#e74c3c","#8e44ad"]
fig, axes = plt.subplots(2, 2, figsize=(10, 9), facecolor="white")
plt.rcParams.update({"font.family":"serif","font.size":10})

# ── (a) Predicted vs Observed ──
ax = axes[0,0]
for z in range(K):
    mask = zones_cal==z
    actual = y_cal[mask]
    pred = np.array([calibrate(z,*env_cal[i]) for i in np.where(mask)[0]])
    ax.scatter(actual, pred, c=ZC[z], s=70, zorder=3,
               label=f"Region {z}", edgecolors="white", linewidth=0.8)
ax.plot([0,1],[0,1],"k--",alpha=0.3,lw=1)
ax.set_xlabel("Observed mean intensity"); ax.set_ylabel("Predicted intensity")
ax.set_title("(a) Predicted vs. observed", fontweight="bold", fontsize=11)
ax.legend(fontsize=8, framealpha=0.9); ax.grid(alpha=0.15)
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_aspect("equal")

# ── (b) Coefficient importance ──
ax = axes[0,1]
names_short = ["Temp", "Solar", "Wind", "RH", "T×S"]
x_pos = np.arange(len(names_short))
w = 0.8/K
for z in range(K):
    coefs = np.abs(models[z]["coefs"])
    ax.bar(x_pos + z*w, coefs, w, color=ZC[z], label=f"R{z}",
           edgecolor="white", linewidth=0.5)
ax.set_xticks(x_pos + w*K/2); ax.set_xticklabels(names_short)
ax.set_ylabel("|Standardized coefficient|")
ax.set_title("(b) Variable importance by region", fontweight="bold", fontsize=11)
ax.legend(fontsize=8, ncol=K); ax.grid(alpha=0.15, axis="y")

# ── (c) Temperature sensitivity ──
ax = axes[1,0]
base = flight_envs.mean(axis=0)
t_range = np.linspace(25, 95, 60)
for z in range(K):
    preds = [calibrate(z,t,base[1],base[2],base[3]) for t in t_range]
    ax.plot(t_range, preds, color=ZC[z], lw=2.2, label=f"Region {z}")
    mask = zones_cal==z
    ax.scatter(env_cal[mask,0], y_cal[mask], c=ZC[z], s=50, zorder=3,
               edgecolors="white", linewidth=0.6)
ax.set_xlabel("Ambient temperature (°F)")
ax.set_ylabel("Calibrated intensity")
ax.set_title("(c) Temperature response", fontweight="bold", fontsize=11)
ax.legend(fontsize=8); ax.grid(alpha=0.15)

# ── (d) Solar sensitivity ──
ax = axes[1,1]
s_range = np.linspace(0, 900, 60)
for z in range(K):
    preds = [calibrate(z,base[0],s,base[2],base[3]) for s in s_range]
    ax.plot(s_range, preds, color=ZC[z], lw=2.2)
    mask = zones_cal==z
    ax.scatter(env_cal[mask,1], y_cal[mask], c=ZC[z], s=50, zorder=3,
               edgecolors="white", linewidth=0.6)
ax.set_xlabel("Solar irradiance (W/m²)")
ax.set_ylabel("Calibrated intensity")
ax.set_title("(d) Solar irradiance response", fontweight="bold", fontsize=11)
ax.grid(alpha=0.15)

plt.suptitle("Environment-conditioned regional calibration model",
             fontsize=13, fontweight="bold", y=1.01, fontfamily="serif")
plt.tight_layout()
plt.savefig("fig_calibration_validation.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig("fig_calibration_validation.pdf", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: fig_calibration_validation.png/pdf")


### Figure B: 3D Thermal Simulation Under Varying Environmental Conditions

In [ ]:
# ============================================================
# FIGURE: 3D SIMULATION COMPARISON (2×3 grid)
# ============================================================

# Precompute zone variation for local texture preservation
zvars = np.zeros(len(zones))
for z in range(K):
    m = zones==z
    if m.any(): zvars[m] = tp[m] - tp[m].mean()

# Height normalization for solar/wind effects
z_norm = (zp_pts - zp_pts.min()) / (zp_pts.max() - zp_pts.min() + 1e-8)

def sim_colors(temp, solar, wind, rh, wind_dir=0):
    """Full simulation: calibration function + spatial physics."""
    # Base: calibration function gives per-zone intensity
    zone_intensities = calibrate_all(temp, solar, wind, rh)
    colors = zone_intensities[zones].copy()

    # Spatial modulation: wind direction (cooling on windward face)
    ref_w = 5.0
    dw = wind - ref_w
    if abs(dw) > 0.5:
        wind_rad = np.radians(wind_dir)
        exposure = np.cos(wind_rad)*xp + np.sin(wind_rad)*yp
        exposure_n = (exposure - exposure.min()) / (exposure.max()-exposure.min()+1e-8)
        wind_cooling = -0.0012 * dw * exposure_n * (0.5 + 0.5*z_norm)
        colors += wind_cooling

    # Spatial modulation: solar height bias (roof gets more)
    ref_s = 550.0
    ds = solar - ref_s
    if abs(ds) > 10:
        solar_height = 0.00008 * ds * z_norm * 0.5
        colors += solar_height

    # Preserve per-point variation within zones
    colors += zvars * 0.30
    return np.clip(colors, 0, 1)


conditions = [
    {"l":"(a) Cold morning\nOvercast, NE wind",   "t":32,"s":120,"w":12,"h":80,"wd":45},
    {"l":"(b) Mild overcast\nCalm, moderate",      "t":55,"s":250,"w":3, "h":65,"wd":0},
    {"l":"(c) Spring afternoon\nSW breeze",        "t":65,"s":600,"w":8, "h":40,"wd":225},
    {"l":"(d) Warm & sunny\nLight S wind",         "t":78,"s":850,"w":5, "h":30,"wd":180},
    {"l":"(e) Hot & humid\nCalm",                  "t":90,"s":700,"w":2, "h":90,"wd":0},
    {"l":"(f) Hot & dry\nStrong W wind, full sun",  "t":92,"s":900,"w":18,"h":25,"wd":270},
]

fig = plt.figure(figsize=(18, 11.5), facecolor="white")
step = max(1, len(xp)//70000)
cmap = cm.inferno

for i, c in enumerate(conditions):
    ax = fig.add_subplot(2, 3, i+1, projection="3d", facecolor="white")
    colors = sim_colors(c["t"], c["s"], c["w"], c["h"], c["wd"])
    rgba = cmap(colors[::step])
    ax.scatter(xp[::step], yp[::step], zp_pts[::step],
               c=rgba, s=0.4, alpha=0.85, rasterized=True, linewidths=0)

    dirs={0:"N",45:"NE",90:"E",135:"SE",180:"S",225:"SW",270:"W",315:"NW"}
    wd_str=dirs.get(c["wd"],f"{c['wd']}°")
    ax.set_title(c["l"], fontsize=10, fontweight="bold", pad=6, fontfamily="serif")
    ax.text2D(0.5,-0.02,
              f"T={c['t']}°F   S={c['s']}W/m²   W={c['w']}mph {wd_str}   RH={c['h']}%",
              transform=ax.transAxes, ha="center", fontsize=7.5,
              color="#555555", fontfamily="monospace")
    ax.view_init(elev=32, azim=55)
    ax.set_box_aspect([1,1,0.35])
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.xaxis.pane.fill=False;ax.yaxis.pane.fill=False;ax.zaxis.pane.fill=False
    ax.xaxis.pane.set_edgecolor("white");ax.yaxis.pane.set_edgecolor("white");ax.zaxis.pane.set_edgecolor("white")
    ax.grid(False)

sm=plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(0,1)); sm.set_array([])
cbar_ax=fig.add_axes([0.25,0.06,0.5,0.015])
cbar=fig.colorbar(sm,cax=cbar_ax,orientation="horizontal")
cbar.set_label("Calibrated thermal intensity",fontsize=10,fontfamily="serif")
cbar.set_ticks([0,0.2,0.4,0.6,0.8,1.0]); cbar.ax.tick_params(labelsize=8)

fig.suptitle("Environment-conditioned thermal simulation of Richard Weeks Hall",
             fontsize=14,fontweight="bold",y=0.97,fontfamily="serif")
fig.text(0.5,0.02,
         "Per-region thermal intensity predicted by calibration model with directional "
         "wind and height-dependent solar modulation.",
         ha="center",fontsize=8.5,color="#666",fontfamily="serif",style="italic")

plt.savefig("fig_3d_simulation.png",dpi=300,bbox_inches="tight",facecolor="white",pad_inches=0.3)
plt.savefig("fig_3d_simulation.pdf",bbox_inches="tight",facecolor="white",pad_inches=0.3)
plt.show()
print("Saved: fig_3d_simulation.png/pdf")


### Figure C: Wind Direction Sensitivity Study
Same conditions, 4 wind directions — shows directional cooling effect on building faces.

In [ ]:
# ============================================================
# FIGURE: WIND DIRECTION COMPARISON (1×4)
# ============================================================

fig = plt.figure(figsize=(18, 5), facecolor="white")
wind_dirs = [0, 90, 180, 270]
dir_names = ["North", "East", "South", "West"]

for i, (wd, dn) in enumerate(zip(wind_dirs, dir_names)):
    ax = fig.add_subplot(1, 4, i+1, projection="3d", facecolor="white")
    colors = sim_colors(65, 500, 15, 40, wd)
    rgba = cmap(colors[::step])
    ax.scatter(xp[::step], yp[::step], zp_pts[::step],
               c=rgba, s=0.35, alpha=0.85, rasterized=True, linewidths=0)
    ax.set_title(f"({chr(97+i)}) Wind from {dn}\n{dn} face cooled",
                 fontsize=10, fontweight="bold", fontfamily="serif", pad=4)
    ax.view_init(elev=30, azim=55)
    ax.set_box_aspect([1,1,0.35])
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.xaxis.pane.fill=False;ax.yaxis.pane.fill=False;ax.zaxis.pane.fill=False
    ax.xaxis.pane.set_edgecolor("white");ax.yaxis.pane.set_edgecolor("white");ax.zaxis.pane.set_edgecolor("white")
    ax.grid(False)

sm2=plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(0,1));sm2.set_array([])
cbar_ax2=fig.add_axes([0.3,0.08,0.4,0.02])
cbar2=fig.colorbar(sm2,cax=cbar_ax2,orientation="horizontal")
cbar2.set_label("Calibrated thermal intensity",fontsize=9); cbar2.ax.tick_params(labelsize=7)

fig.suptitle("Wind direction sensitivity — T=65°F, Solar=500 W/m², Wind=15 mph, RH=40%",
             fontsize=12, fontweight="bold", y=1.02, fontfamily="serif")
plt.tight_layout(rect=[0,0.12,1,0.95])
plt.savefig("fig_wind_direction.png",dpi=300,bbox_inches="tight",facecolor="white")
plt.savefig("fig_wind_direction.pdf",bbox_inches="tight",facecolor="white")
plt.show()
print("Saved: fig_wind_direction.png/pdf")


### Summary: The Calibration Equations
For reference in the paper — these are the fitted model parameters.

In [ ]:
# ============================================================
# CALIBRATION EQUATION SUMMARY (for paper)
# ============================================================

print("ENVIRONMENT-CONDITIONED REGIONAL CALIBRATION MODEL")
print("="*70)
print()
print("General form (per region k):")
print("  Ŷ_k = β₀ + β₁·T̂ + β₂·Ŝ + β₃·Ŵ + β₄·Ĥ + β₅·(T̂·Ŝ)")
print()
print("where X̂ = (X - μ_X) / σ_X  (standardized)")
print()

for z in range(K):
    m = models[z]
    sc = m["scaler"]
    print(f"Region {z}: {zone_labels[z]}")
    print(f"  R² = {m['r2']:.4f}    RMSE = {m['rmse']:.6f}")
    print(f"  β₀ = {m['intercept']:.6f}")
    for j, name in enumerate(["T_amb","Solar","Wind","RH","T×S"]):
        print(f"  β_{j+1}({name:5s}) = {m['coefs'][j]:+.6f}   "
              f"[μ={sc.mean_[j]:.1f}, σ={sc.scale_[j]:.1f}]")
    print()

# Export as JSON for portability
export = {"model":"per_region_ridge_with_interaction", "n_regions":K}
export["regions"] = {}
for z in range(K):
    m = models[z]
    export["regions"][str(z)] = {
        "label": zone_labels[z],
        "intercept": float(m["intercept"]),
        "coefficients": dict(zip(["temp","solar","wind","rh","temp_x_solar"],
                                 m["coefs"].tolist())),
        "scaler_mean": m["scaler"].mean_.tolist(),
        "scaler_std": m["scaler"].scale_.tolist(),
        "r2": float(m["r2"]),
        "rmse": float(m["rmse"]),
    }
with open("calibration_model.json","w") as f:
    json.dump(export, f, indent=2)
print("Exported: calibration_model.json")


In [ ]:
# ============================================================
# INTERACTIVE 3D — change values and re-run
# ============================================================

# ── CHANGE THESE VALUES ──
TEMP = 70       # °F
SOLAR = 600     # W/m²
WIND = 8        # mph
RH = 45         # %
WIND_DIR = 180  # degrees (0=N, 90=E, 180=S, 270=W)
# ──────────────────────────

colors = sim_colors(TEMP, SOLAR, WIND, RH, WIND_DIR)
dirs={0:"N",45:"NE",90:"E",135:"SE",180:"S",225:"SW",270:"W",315:"NW"}

fig = go.Figure(data=[go.Scatter3d(
    x=xp, y=yp, z=zp_pts, mode="markers",
    marker=dict(size=1.2, color=colors, colorscale="Inferno",
                cmin=0, cmax=1,
                colorbar=dict(title="Thermal", thickness=12, len=0.5),
                opacity=0.85),
    hovertemplate="X:%{x:.0f} Y:%{y:.0f} Z:%{z:.0f}<extra></extra>"
)])
fig.update_layout(
    title=f"T={TEMP}°F | Solar={SOLAR} | Wind={WIND}mph {dirs.get(WIND_DIR,'')} | RH={RH}%",
    scene=dict(xaxis_title="X(m)",yaxis_title="Y(m)",zaxis_title="Z(m)",
               aspectmode="data",camera=dict(eye=dict(x=1.5,y=1.5,z=1.0))),
    width=900,height=620,template="plotly_dark",margin=dict(l=0,r=0,t=40,b=0))
fig.show()
print("Change TEMP, SOLAR, WIND, RH, WIND_DIR above and re-run.")


In [ ]:
# 3D simulation: calibration + spatial physics
def simulate(temp, solar, wind, rh, wind_dir):
    # Step 1: Calibration model predicts per-region intensity
    zone_intensities = [calibrate(z, temp, solar, wind, rh)
                        for z in range(K)]
    colors = zone_intensities[point_zones]

    # Step 2: Directional wind cooling (windward vs leeward)
    exposure = cos(wind_dir)*x + sin(wind_dir)*y
    colors += -0.0012 * (wind-5) * normalize(exposure)

    # Step 3: Height-weighted solar (roof > ground)
    colors += 0.00008 * (solar-550) * height_normalized

    return clip(colors, 0, 1)